# SOLID principles

## SRP - Single Responsibility Principle 
- only one reason to change
- one responsibility

## OCP - Open/Closed Principle
- open for extension closed for modification
- when adding new features - it shouldn't change existing, tested code

## LSP - Liskov Substitution Principle
- superclass instances should be replaceable with subclass instances without breaking correctness

## ISP - Interface Segregation Principle
- clients not depend on methods they don't use
- split large interfaces into smaller 

## DIP - Dependency Inversion Principle
- high level modules should not depend on low level modules
- both should depend on abstractions



Important question to ask
- what is the abstraction?

## Violating principles

### SRP 

reasons to change:
- business rules (tax logic, salary validation)
- report layout 
- file handling changes

### OCP

add PDF or HTML report 
- modify generate_report

### DIP
Employee (high lvl) depends on low lvl modules:
- json module
- open(...)

In [ ]:
from numbers import Real
import json


class Employee:
    def __init__(self, name, phone, salary):
        self.name = name
        self.phone = phone
        self.salary = salary

    @property
    def salary(self):
        return self._salary

    @salary.setter
    def salary(self, salary):
        if not isinstance(salary, Real):
            raise TypeError(f"salary must be a number not {type(salary).__name__}")

        if salary < 0:
            raise ValueError(f"Salary must be 0 or higher not {salary}")

        self._salary = salary

    # responsibility 1: business logic
    def net_salary(self, tax_rate=0.3):
        return self.salary * (1 - tax_rate)

    # responsibility 2: reporting
    # violates OCP, need to modify to add one more report type
    def generate_report(self, report_type="txt"):
        if report_type == "txt":
            with open(f"{self.name}.txt", "w") as file:
                file.write("Employee card\n")
                file.write(
                    f"Name: {self.name}\nSalary: {self.salary}\nPhone: {self.phone}"
                )
        elif report_type == "json":
            with open(f"{self.name}.json", "w") as file:
                json.dump(
                    {
                        "name": self.name,
                        "phone": self.phone,
                        "salary": self.salary,
                        "net_salary": self.net_salary(),
                    },
                    file,
                    indent=2,
                )

    def __repr__(self):
        return f"Employee({self.name}, {self.phone}, {self.salary})"


e1 = Employee("Anders Eriksson", "038-01480", 53_000)
e1.generate_report("txt")
e1


Employee(Anders Eriksson, 038-01480, 53000)

## Following principles

### SRP
- Employee has data and validation and net_salary but it's part of one abstraction -> one concern one reason to change 

### OCP
- If we add a pdf report, before we would need to modify the generate_report method which violates OCP (closed for modification)
- now we can extend by creating another class PdfReport
- this makes testing easier

### LSP
- both TxtReport and JsonReport are interchangeable when Report is expected
- they honor the contract to return a string

### ISP
- Report has an abstract method: content
- all implementers (TxtReport and JsonReport) is only forced to implement content method which it needs, and nothing that they don't need

### DIP
- Employee (high lvl) knows nothing about Report
- Report (low lvl) depends on Employee

lvl - distance from core business concept or I/O


In [17]:
from numbers import Real
import json
from abc import ABC, abstractmethod


class Employee:
    def __init__(self, name, phone, salary, tax_rate=0.3):
        self.name = name
        self.phone = phone
        self.salary = salary
        self.tax_rate = tax_rate

    @property
    def net_salary(self):
        return self.salary * (1 - self.tax_rate)

    @property
    def salary(self):
        return self._salary

    @salary.setter
    def salary(self, salary):
        if not isinstance(salary, Real):
            raise TypeError(f"salary must be a number not {type(salary).__name__}")

        if salary < 0:
            raise ValueError(f"Salary must be 0 or higher not {salary}")

        self._salary = salary

    def __repr__(self):
        return f"Employee({self.name}, {self.phone}, {self.salary})"


class Report(ABC):
    @abstractmethod
    def content(self, employee: Employee) -> str: ...


class TxtReport(Report):
    def content(self, employee: Employee) -> str:
        return (
            "Employee card\n"
            f"Name: {employee.name}\nSalary: {employee.salary}\nNet salary: {employee.net_salary}\nPhone: {employee.phone}"
        )


class JsonReport(Report):
    def content(self, employee) -> str:
        return json.dumps(
            {
                "name": employee.name,
                "phone": employee.phone,
                "salary": employee.salary,
                "net_salary": employee.net_salary,
            }, indent=2
        )

def export(text, path):
    with open(path, "w") as file:
        file.write(text)

e1 = Employee("Anders Eriksson", "038-01480", 53_000, tax_rate=.42)
txt_report = TxtReport()
export(txt_report.content(e1), "Anders.txt")

json_report = JsonReport()
export(json_report.content(e1), "Anders.json")

e1

Employee(Anders Eriksson, 038-01480, 53000)